# NLP Assignment 1: Full-Corpus Colab Build Pipeline

**Course:** Natural Language Processing | Addis Ababa University  
**Corpus:** Amharic Corpus (GPAC) — Full 5.0M Sentences / 77M Tokens  

This notebook executes the heavy build and evaluation stages on Google Colab under **AGENTS.md Rule B13**:
1. Mounts Google Drive and stages repository in `/content/NLP_Assignment1`
2. Installs pinned dependencies (`numpy`, `psutil`, `wordcloud`, `scikit-learn`)
3. Sequentially executes `scripts/01` to `scripts/13` with SHA-256 checkpointing
4. Writes `data/checksums.json`
5. Exports `data/` and `results/` to Google Drive

### Cell 1: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### Cell 2: Stage Repository and Configure Working Directory
Update `DRIVE_REPO_PATH` to point to the repository folder on your Google Drive.

In [ ]:
import os
import shutil
from pathlib import Path

# Set your Drive path to the repository
DRIVE_REPO_PATH = Path('/content/drive/MyDrive/Assignment 1')
LOCAL_WORKDIR = Path('/content/NLP_Assignment1')

if not LOCAL_WORKDIR.exists():
    print(f"Copying repo from {DRIVE_REPO_PATH} to {LOCAL_WORKDIR}...")
    shutil.copytree(DRIVE_REPO_PATH, LOCAL_WORKDIR, ignore=shutil.ignore_patterns('.git', 'node_modules', '.next'))
    print("Repository staged successfully.")
else:
    print(f"Workdir already exists at {LOCAL_WORKDIR}")

os.chdir(LOCAL_WORKDIR)
print(f"Current working directory: {os.getcwd()}")

### Cell 3: Install Pinned Dependencies and Check Runtime Info

In [ ]:
!pip install -q numpy psutil wordcloud scikit-learn

import psutil
import numpy as np
import sklearn
import wordcloud

ram_gb = psutil.virtual_memory().total / (1024**3)
print(f"Colab Environment Info:")
print(f"  Total System RAM: {ram_gb:.2f} GB")
print(f"  Python Version:   {sys.version.split()[0]}")
print(f"  NumPy Version:    {np.__version__}")
print(f"  Scikit-Learn:     {sklearn.__version__}")
print(f"  Wordcloud:        {wordcloud.__version__}")

### Cell 4: Execute Heavy Build Pipeline (Scripts 01 through 13)
Stages with completed checkpoints in `data/checksums.json` are automatically skipped.

In [ ]:
import subprocess
import time

pipeline_scripts = [
    "scripts/01_assess_raw.py",
    "scripts/02_build_cleaned.py",
    "scripts/03_verify_cleaned.py",
    "scripts/04_build_vocab_tokens.py",
    "scripts/05_profile_counts.py",
    "scripts/06_split.py",
    "scripts/07_count_full.py",
    "scripts/08_count_train.py",
    "scripts/10_q1_results.py",
    "scripts/11_q2_perplexity.py",
    "scripts/12_q3_extrinsic.py",
    "scripts/13_q4_eval.py",
]

for script_path in pipeline_scripts:
    print(f"\n{'='*30} Running {script_path} {'='*30}")
    t0 = time.time()
    res = subprocess.run(["python", script_path], check=True)
    print(f"Finished {script_path} in {time.time() - t0:.2f} s")

### Cell 5: Verify and Display Final Checksums

In [ ]:
import json

checksums_file = Path('data/checksums.json')
if checksums_file.exists():
    with open(checksums_file, 'r', encoding='utf-8') as f:
        data = json.load(f)
    print(f"Recorded Checksums ({len(data)} total artifacts):")
    for k, v in sorted(data.items()):
        print(f"  {k:45s} -> {v}")
else:
    print("[WARN] Checksums file not generated.")

### Cell 6: Sync Artifacts Back to Google Drive
Saves `results/`, `data/counts/train_seed42/`, and `data/checksums.json` to Google Drive for local PC download.

In [ ]:
DRIVE_ARTIFACTS_DIR = Path('/content/drive/MyDrive/NLP_Assignment1_artifacts')
DRIVE_ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Syncing results and training counts to {DRIVE_ARTIFACTS_DIR}...")

# Sync results
shutil.copytree('results', DRIVE_ARTIFACTS_DIR / 'results', dirs_exist_ok=True)

# Sync checksums and splits
shutil.copy('data/checksums.json', DRIVE_ARTIFACTS_DIR / 'checksums.json')
for f in Path('data').glob('split_indices_*.json'):
    shutil.copy(f, DRIVE_ARTIFACTS_DIR / f.name)

# Sync training split counts for local PC inference
train_counts_src = Path('data/counts/train_seed42')
if train_counts_src.exists():
    shutil.copytree(train_counts_src, DRIVE_ARTIFACTS_DIR / 'counts/train_seed42', dirs_exist_ok=True)

print("Artifact sync complete! You can now download NLP_Assignment1_artifacts/ to your local PC repository.")